In [ ]:
import os
import pandas as pd

from socceraction.data.statsbomb import StatsBombLoader

from football_ai.data import (
    build_and_save_vaep_for_competition_season,
    list_available_dataset_keys,
    load_xy,
    resolve_competition_season_ids,
    select_competition_seasons,
    slug,
    split_dataset_key,
)
from football_ai.evaluation import evaluate_binary, get_positive_class_scores

pd.set_option('display.max_rows', 10)

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
# All data pipeline functions are imported from football_ai.data:
#   slug, resolve_competition_season_ids, build_and_save_vaep_for_competition_season, etc.
# All evaluation functions are imported from football_ai.evaluation:
#   get_positive_class_scores, evaluate_binary

In [3]:
# Paths
data_root = '../../open-data/data'  # StatsBomb open-data JSON folder
output_dir = os.path.join('..', 'data', 'spadl_data_major_leagues')
os.makedirs(output_dir, exist_ok=True)

# Loader
SBL = StatsBombLoader(getter='local', root=data_root)

print('output_dir:', output_dir)

output_dir: ../data/spadl_data_major_leagues


In [4]:
# List available competitions/seasons
df_competitions = SBL.competitions().copy()
cols = ['competition_id', 'season_id', 'competition_name', 'season_name']
display(df_competitions[cols].sort_values(['competition_name', 'season_name']).reset_index(drop=True))

,competition_id,season_id,competition_name,season_name
0,9,27,1. Bundesliga,2015/2016
1,9,281,1. Bundesliga,2023/2024
2,1267,107,African Cup of Nations,2023
3,16,276,Champions League,1970/1971
4,16,71,Champions League,1971/1972
...,...,...,...,...
70,35,75,UEFA Europa League,1988/1989
71,53,106,UEFA Women's Euro,2022
72,53,315,UEFA Women's Euro,2025
73,72,30,Women's World Cup,2019


In [5]:
for el in df_competitions['competition_name'].value_counts().index:
    print(el)


Champions League
La Liga
FIFA World Cup
Copa del Rey
Ligue 1
FA Women's Super League
Women's World Cup
1. Bundesliga
UEFA Euro
Liga Profesional
UEFA Women's Euro
Premier League
Serie A
Copa America
African Cup of Nations
FIFA U20 World Cup
Indian Super league
Major League Soccer
NWSL
North American League
UEFA Europa League


In [6]:
selected_leagues = ["La Liga", "Serie A", "Premier League", "1. Bundesliga", "Ligue 1", "Champions League", "UEFA Europa League"]
# women_leagues = ["FA Women's Super League", "Women's World Cup", "UEFA Women's Euro"]
selected_leagues_mask = df_competitions['competition_name'].isin(selected_leagues)
# selected_leagues_mask = df_competitions['competition_name'].isin(women_leagues)
df_competitions = df_competitions[selected_leagues_mask]

In [7]:
df_competitions.value_counts(subset=['competition_name'])

competition_name  
Champions League      18
La Liga               18
Ligue 1                3
1. Bundesliga          2
Premier League         2
Serie A                2
UEFA Europa League     1
Name: count, dtype: int64

## Select league/season

Edit `selected_name_pairs` to include the league(s) and season(s) you want.

- The `competition_name` and `season_name` must match exactly the values shown in the table above.
- If there are duplicates, use `selected_id_pairs` instead (competition_id, season_id).

In [8]:
# Select league/season pairs to process
SAVE_ALL_AVAILABLE = True  # True => process every available competition+season in df_selected_competitions

# OPTION A: select by names (used only when SAVE_ALL_AVAILABLE = False)
selected_name_pairs = [
    # ('Serie A', '2015/2016'),
]

# OPTION B: select by ids (used only when SAVE_ALL_AVAILABLE = False)
selected_id_pairs = [
    # (11, 27),
]

if SAVE_ALL_AVAILABLE:
    selected = [
        (int(row.competition_id), int(row.season_id))
        for row in df_competitions[['competition_id', 'season_id']].drop_duplicates().itertuples(index=False)
    ]
elif selected_name_pairs and selected_id_pairs:
    raise ValueError('Use either selected_name_pairs OR selected_id_pairs (not both).')
elif selected_name_pairs:
    selected = [resolve_competition_season_ids(df_competitions, c, s) for (c, s) in selected_name_pairs]
elif selected_id_pairs:
    selected = [(int(c), int(s)) for (c, s) in selected_id_pairs]
else:
    selected = []

print('Total selected (competition_id, season_id):', len(selected))
print('Preview:', selected[:10])

Total selected (competition_id, season_id): 46
Preview: [(9, 281), (9, 27), (16, 4), (16, 1), (16, 2), (16, 27), (16, 26), (16, 25), (16, 24), (16, 23)]


In [ ]:
# Build and save one dataset per selected competition/season
if not selected:
    raise ValueError('No league/season selected. Configure SAVE_ALL_AVAILABLE or selection lists above.')

outputs = []
failed = []

for competition_id, season_id in selected:
    try:
        fpath, lpath, meta = build_and_save_vaep_for_competition_season(
            loader=SBL,
            competitions_df=df_competitions,
            output_dir=output_dir,
            competition_id=competition_id,
            season_id=season_id,
        )
        outputs.append((fpath, lpath))
        if meta.get('games_failed'):
            print(f'  Warning: {meta["games_failed"]} games failed')
    except Exception as e:
        failed.append((competition_id, season_id, str(e)))
        print(f'Skipped (cid={competition_id}, sid={season_id}): {e}')

print('\nDone.')
print('Saved datasets:', len(outputs))
print('Failed datasets:', len(failed))

if outputs:
    print('\nSaved files:')
    for fpath, lpath in outputs:
        print('-', fpath)
        print('-', lpath)

if failed:
    print('\nFirst failed items:')
    display(pd.DataFrame(failed, columns=['competition_id', 'season_id', 'error']).head(10))


=== 1. Bundesliga | 2023/2024 (cid=9, sid=281) ===
-> ../data/spadl_data_major_leagues/features_1_bundesliga_2023_2024.h5
-> ../data/spadl_data_major_leagues/labels_1_bundesliga_2023_2024.h5
Saved features: (80212, 38)
Saved labels:   (80212, 4)

=== 1. Bundesliga | 2015/2016 (cid=9, sid=27) ===
-> ../data/spadl_data_major_leagues/features_1_bundesliga_2015_2016.h5
-> ../data/spadl_data_major_leagues/labels_1_bundesliga_2015_2016.h5
converted 50/306 games
converted 100/306 games
converted 150/306 games
converted 200/306 games
converted 250/306 games
converted 300/306 games
Saved features: (596336, 38)
Saved labels:   (596336, 4)

=== Champions League | 2018/2019 (cid=16, sid=4) ===
-> ../data/spadl_data_major_leagues/features_champions_league_2018_2019.h5
-> ../data/spadl_data_major_leagues/labels_champions_league_2018_2019.h5
Saved features: (1793, 38)
Saved labels:   (1793, 4)

=== Champions League | 2017/2018 (cid=16, sid=1) ===
-> ../data/spadl_data_major_leagues/features_champion

In [ ]:
trainval_leagues = ["La Liga", "Serie A", "Premier League", "1. Bundesliga", "Ligue 1"]
test_leagues = ["Champions League", "UEFA Europa League"]

selected_trainval = df_competitions[df_competitions.competition_name.isin(trainval_leagues)].value_counts(subset=['competition_id', 'season_id']).index.tolist()
selected_test = df_competitions[df_competitions.competition_name.isin(test_leagues)].value_counts(subset=['competition_id', 'season_id']).index.tolist()

In [ ]:
# Train a RandomForest on ALL selected data (no split)
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Collect (features, labels) from saved files for each selected comp/season
df_all_train = []
for competition_id, season_id in selected_trainval:
    row = df_competitions[(df_competitions.competition_id == competition_id) & (df_competitions.season_id == season_id)]
    if len(row) != 1:
        raise ValueError(f'Cannot uniquely resolve competition_id={competition_id}, season_id={season_id}')
    comp_name = str(row.iloc[0].competition_name)
    season_name = str(row.iloc[0].season_name)
    league_slug = slug(comp_name)
    season_slug = slug(season_name)
    features_path = os.path.join(output_dir, f'features_{league_slug}_{season_slug}.h5')
    labels_path = os.path.join(output_dir, f'labels_{league_slug}_{season_slug}.h5')
    if not (os.path.exists(features_path) and os.path.exists(labels_path)):
        raise FileNotFoundError(
            f'Missing files for {comp_name} {season_name}. Expected:\n- {features_path}\n- {labels_path}\nRun the dataset-generation cell first.'
        )

    with pd.HDFStore(features_path, mode='r') as store:
        df_features = store['features'].copy()
    with pd.HDFStore(labels_path, mode='r') as store:
        df_labels = store['labels'].copy()

    df = df_features.merge(df_labels, on=['game_id', 'action_id'], how='inner')
    df['competition_id'] = competition_id
    df['season_id'] = season_id
    df_all_train.append(df)
    print(f'Loaded + joined {comp_name} {season_name}:', df.shape)

df_all_train = pd.concat(df_all_train, ignore_index=True)
print('\nTotal joined dataset:', df_all_train.shape)

# Build X/y (no split)
drop_cols = {'game_id', 'action_id', 'scores', 'concedes', 'competition_id', 'season_id'}
feature_cols = [c for c in df_all_train.columns if c not in drop_cols]
X = df_all_train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_scores = df_all_train['scores'].astype(int)
y_concedes = df_all_train['concedes'].astype(int)

print('X:', X.shape, 'y_scores:', y_scores.shape, 'y_concedes:', y_concedes.shape)
for c in feature_cols:
    print(f'- {c}: {X[c].dtype}, min={X[c].min()}, max={X[c].max()}')

print('Score and concede rates:')
print('Score rate:', y_scores.sum() / y_scores.shape[0])
print('Concede rate:', y_concedes.sum() / y_concedes.shape[0])

# RandomForest (baseline)
rf_scores = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced_subsample',
)

rf_scores.fit(X, y_scores)

print('Trained RandomForest models on full dataset (no split).')

Loaded + joined Premier League 2015/2016: (758427, 42)
Loaded + joined Premier League 2003/2004: (74759, 42)
Loaded + joined Ligue 1 2015/2016: (766059, 42)
Loaded + joined Ligue 1 2021/2022: (60579, 42)
Loaded + joined Ligue 1 2022/2023: (76367, 42)
Loaded + joined La Liga 2017/2018: (81134, 42)
Loaded + joined La Liga 2016/2017: (72987, 42)
Loaded + joined La Liga 2018/2019: (77533, 42)
Loaded + joined La Liga 2009/2010: (75211, 42)
Loaded + joined La Liga 2010/2011: (77494, 42)
Loaded + joined La Liga 2011/2012: (85924, 42)
Loaded + joined La Liga 2012/2013: (77112, 42)
Loaded + joined La Liga 2013/2014: (69202, 42)
Loaded + joined La Liga 2014/2015: (84209, 42)
Loaded + joined La Liga 2015/2016: (757300, 42)
Loaded + joined La Liga 2004/2005: (13519, 42)
Loaded + joined La Liga 2005/2006: (33540, 42)
Loaded + joined La Liga 2006/2007: (54543, 42)
Loaded + joined La Liga 2007/2008: (59539, 42)
Loaded + joined La Liga 2008/2009: (62727, 42)
Loaded + joined La Liga 2019/2020: (75621, 

In [ ]:
# Build test set datasets
print('Selected (competition_id, season_id):', selected_test)

if not selected_test:
    raise ValueError('No league/season selected. Edit selected_name_pairs or selected_id_pairs above.')

outputs = []
for competition_id, season_id in selected_test:
    fpath, lpath, meta = build_and_save_vaep_for_competition_season(
        loader=SBL,
        competitions_df=df_competitions,
        output_dir=output_dir,
        competition_id=competition_id,
        season_id=season_id,
    )
    outputs.append((fpath, lpath))

print('\nDone. Outputs:')
for fpath, lpath in outputs:
    print('-', fpath)
    print('-', lpath)

Selected (competition_id, season_id): [(16, 1), (16, 2), (16, 4), (16, 21), (16, 22), (16, 23), (16, 24), (16, 25), (16, 26), (16, 27), (16, 37), (16, 39), (16, 41), (16, 44), (16, 71), (16, 76), (16, 276), (16, 277)]

=== Champions League | 2017/2018 (cid=16, sid=1) ===
-> ../data/spadl_data_major_leagues/features_champions_league_2017_2018.h5
-> ../data/spadl_data_major_leagues/labels_champions_league_2017_2018.h5
Saved features: (2052, 38)
Saved labels:   (2052, 4)

=== Champions League | 2016/2017 (cid=16, sid=2) ===
-> ../data/spadl_data_major_leagues/features_champions_league_2016_2017.h5
-> ../data/spadl_data_major_leagues/labels_champions_league_2016_2017.h5
Saved features: (2062, 38)
Saved labels:   (2062, 4)

=== Champions League | 2018/2019 (cid=16, sid=4) ===
-> ../data/spadl_data_major_leagues/features_champions_league_2018_2019.h5
-> ../data/spadl_data_major_leagues/labels_champions_league_2018_2019.h5
Saved features: (1793, 38)
Saved labels:   (1793, 4)

=== Champions Le

In [ ]:
# Collect (features, labels) from saved files for each selected comp/season
import numpy as np

df_all_test = []
for competition_id, season_id in selected_test:
    row = df_competitions[(df_competitions.competition_id == competition_id) & (df_competitions.season_id == season_id)]
    if len(row) != 1:
        raise ValueError(f'Cannot uniquely resolve competition_id={competition_id}, season_id={season_id}')
    comp_name = str(row.iloc[0].competition_name)
    season_name = str(row.iloc[0].season_name)
    league_slug = slug(comp_name)
    season_slug = slug(season_name)
    features_path = os.path.join(output_dir, f'features_{league_slug}_{season_slug}.h5')
    labels_path = os.path.join(output_dir, f'labels_{league_slug}_{season_slug}.h5')
    if not (os.path.exists(features_path) and os.path.exists(labels_path)):
        raise FileNotFoundError(
            f'Missing files for {comp_name} {season_name}. Expected:\n- {features_path}\n- {labels_path}\nRun the dataset-generation cell first.'
        )

    with pd.HDFStore(features_path, mode='r') as store:
        df_features = store['features'].copy()
    with pd.HDFStore(labels_path, mode='r') as store:
        df_labels = store['labels'].copy()

    df = df_features.merge(df_labels, on=['game_id', 'action_id'], how='inner')
    df['competition_id'] = competition_id
    df['season_id'] = season_id
    df_all_test.append(df)
    print(f'Loaded + joined {comp_name} {season_name}:', df.shape)

df_test = pd.concat(df_all_test, ignore_index=True)
print('\nTotal joined dataset:', df_test.shape)

# Build X/y (no split)
drop_cols = {'game_id', 'action_id', 'scores', 'concedes', 'competition_id', 'season_id'}
feature_cols = [c for c in df_test.columns if c not in drop_cols]
X = df_test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y_scores = df_test['scores'].astype(int)
y_concedes = df_test['concedes'].astype(int)

print('X:', X.shape, 'y_scores:', y_scores.shape, 'y_concedes:', y_concedes.shape)

Loaded + joined Champions League 2017/2018: (2052, 42)
Loaded + joined Champions League 2016/2017: (2062, 42)
Loaded + joined Champions League 2018/2019: (1793, 42)
Loaded + joined Champions League 2009/2010: (1968, 42)
Loaded + joined Champions League 2010/2011: (2559, 42)
Loaded + joined Champions League 2011/2012: (2770, 42)
Loaded + joined Champions League 2012/2013: (1957, 42)
Loaded + joined Champions League 2013/2014: (2575, 42)
Loaded + joined Champions League 2014/2015: (2082, 42)
Loaded + joined Champions League 2015/2016: (2853, 42)
Loaded + joined Champions League 2004/2005: (2699, 42)
Loaded + joined Champions League 2006/2007: (1763, 42)
Loaded + joined Champions League 2008/2009: (2045, 42)
Loaded + joined Champions League 2003/2004: (1855, 42)
Loaded + joined Champions League 1971/1972: (1819, 42)
Loaded + joined Champions League 1999/2000: (2006, 42)
Loaded + joined Champions League 1970/1971: (2034, 42)
Loaded + joined Champions League 1972/1973: (1812, 42)

Total joi

In [ ]:
# Test the trained RandomForest on this test dataset

# Align feature columns to what the model was trained on (handles missing/extra columns)
if hasattr(rf_scores, 'feature_names_in_'):
    train_cols = list(rf_scores.feature_names_in_)
    X_test_aligned = X.reindex(columns=train_cols, fill_value=0)
else:
    X_test_aligned = X.copy()

# Predict probabilities using library helper
p_scores = get_positive_class_scores(rf_scores, X_test_aligned)

# Evaluate using library helper (returns dict with roc_auc, pr_auc, brier, logloss, precision, recall, f1, ...)
threshold = 0.5
metrics = evaluate_binary(p_scores, y_scores, threshold=threshold)
print('[scores] Evaluation metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.6f}')

# Preview a few predictions alongside labels
yhat_scores = (p_scores >= threshold).astype(int)
df_pred = df_test[['game_id', 'action_id', 'scores', 'concedes']].copy()
df_pred['p_scores'] = p_scores
df_pred['yhat_scores'] = yhat_scores
display(df_pred.head(10))


[scores]
ROC AUC:   0.7595186248471403
PR AUC:    0.23166212146868262
Log loss:  0.11851994821335037
Brier:     0.011376529557668458

[scores:] (threshold=0.5)
Accuracy:  0.9881924348904506
Precision: 0.9696969696969697
Recall:    0.1233140655105973
F1:        0.2188034188034188


,game_id,action_id,scores,concedes,p_scores,yhat_scores
0,18245,0,False,False,0.000,0
1,18245,1,False,False,0.000,0
2,18245,2,False,False,0.000,0
3,18245,3,False,False,0.000,0
4,18245,4,False,False,0.005,0
5,18245,5,False,False,0.000,0
6,18245,6,False,False,0.005,0
7,18245,7,False,False,0.000,0
8,18245,8,False,False,0.000,0
9,18245,9,False,False,0.000,0


In [ ]:
from sklearn.metrics import confusion_matrix

test_cm = confusion_matrix(y_scores, yhat_scores)
df_cm = pd.DataFrame(
    test_cm,
    index=["Actual Negative", "Actual Positive"],
    columns=["Predicted Negative", "Predicted Positive"],
)
tn, fp, fn, tp = test_cm.ravel()
print("\n=== TEST ===")
print(df_cm)
print(f"TP: {tp}, TN: {tn}, FP: {fp}, FN: {fn}")


=== TEST ===
Predicted        Predicted Negative  Predicted Positive
Actual                                                 
Actual Negative               38183                   2
Actual Positive                 455                  64
TP: 64, TN: 38183, FP: 2, FN: 455


Predicted,Predicted Negative,Predicted Positive
Actual,,
Actual Negative,38183,2
Actual Positive,455,64
